# Local Data Agent — free temporary hosting on a Colab GPU

This notebook runs the whole project on this VM and gives you a public link to share:
MySQL with the `employees` sample database → Ollama on the GPU (`qwen2.5-coder:3b` for SQL, `qwen3:4b` thinking model
for the answers and the Expert AI) → trained ML models → the Streamlit app → a Cloudflare quick tunnel (no account).

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then run the cells top to bottom.
First run ≈ 10 minutes (mostly the 4.4 GB model download). The link works while this notebook stays open
(Colab ends sessions after ~90 min idle / ~12 h) and changes on every run.


In [ ]:
#@title 1 · GPU check and get the code
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE  ->  Runtime > Change runtime type > T4 GPU, then re-run")
if not os.path.isdir("/content/database-agent"):
    !git clone -q https://github.com/k642512255032-max/database-agent.git /content/database-agent
%cd /content/database-agent
!git pull -q
!chmod +x deploy/colab/*.sh
print("code ready")


In [ ]:
#@title 2 · Install everything (MySQL + employees DB, Ollama + models, app deps, trained models) — ~10 min
!bash deploy/colab/bootstrap.sh


In [ ]:
#@title 3 · Start the app + public link  (re-run this cell any time something looks down)
!bash deploy/colab/serve.sh


In [ ]:
#@title 4 · Status check (run after cell 3, or whenever the sidebar shows something offline)
import subprocess, re, json, urllib.request
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
print("GPU        :", sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader") or "none")
print("MySQL      :", "running" if sh("mysqladmin ping --silent && echo ok") else "NOT running")
try:
    tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5))
    print("Ollama     : running - models:", ", ".join(m["name"] for m in tags["models"]))
except Exception as e:
    print("Ollama     : NOT running ->", e)
print("Streamlit  :", "running" if sh("curl -fs http://127.0.0.1:8501/_stcore/health") else "NOT running")
log = sh("cat /tmp/agent-logs/cloudflared.log 2>/dev/null")
m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Public link:", m.group(0) if m else "none - re-run cell 3")


In [ ]:
#@title 5 · Keep alive — leave running while people use the link (stop it to end hosting)
import time, subprocess, re
url = ""
try:
    url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/agent-logs/cloudflared.log").read()).group(0)
except Exception:
    pass
while True:
    up = lambda pat: subprocess.run(["pgrep", "-f", pat], capture_output=True).returncode == 0
    state = "OK" if (up("streamlit run app.py") and up("ollama serve") and up("cloudflared tunnel")) else "SOMETHING DOWN - re-run cell 3"
    print(time.strftime("%H:%M:%S"), state, "|", url, flush=True)
    time.sleep(300)


In [ ]:
#@title 6 · Troubleshooting: show the logs
!echo "--- streamlit"; tail -25 /tmp/agent-logs/streamlit.log 2>/dev/null
!echo "--- ollama";    tail -15 /tmp/agent-logs/ollama.log 2>/dev/null
!echo "--- tunnel";    tail -15 /tmp/agent-logs/cloudflared.log 2>/dev/null
!echo "--- training";  tail -8  /tmp/agent-logs/train.log 2>/dev/null


### Notes
* **Netlify publishing** (Dashboards page) is off on this VM — no token is set. Everything else works.
* People with the link can query the sample database (read-only user) and use the models; nothing else on the VM is exposed.
* First answer is slow while the models load into the GPU (~30 s); afterwards a chat turn with the Expert AI takes ~20–60 s on a T4.
* If Cloudflare refuses a tunnel repeatedly, this no-account fallback also gives a public URL:
  `!ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8501 a.pinggy.io`
